In [9]:
import requests
import pandas as pd

BASE_URL = "https://www.backend.ufastats.com/api/v1"

params = {
    "date": "2024-01-01:2024-12-31"
}

response = requests.get(f"{BASE_URL}/games", params=params, timeout=20)

print(response.url)
print(response.status_code)
print(response.text[:500])

response.raise_for_status()

games_json = response.json()

print(type(games_json))
games_json

https://www.backend.ufastats.com/api/v1/games?date=2024-01-01%3A2024-12-31
200
{"object":"list","data":[{"gameID":"2024-05-11-MTL-TOR","awayTeamID":"royal","homeTeamID":"rush","awayScore":15,"homeScore":12,"status":"Final","startTimestamp":"2024-05-11T19:00:00-04:00","startTimezone":"EDT","streamingURL":"https://watchufa.tv/videos/montreal-at-toronto-5-11-2024","updateTimestamp":"2024-05-22T23:31:12.385Z","week":"week-3","location":"Allan A. Lamport Stadium"},{"gameID":"2024-04-27-MIN-PIT","awayTeamID":"windchill","homeTeamID":"thunderbirds","awayScore":18,"homeScore":11,"
<class 'dict'>


{'object': 'list',
 'data': [{'gameID': '2024-05-11-MTL-TOR',
   'awayTeamID': 'royal',
   'homeTeamID': 'rush',
   'awayScore': 15,
   'homeScore': 12,
   'status': 'Final',
   'startTimestamp': '2024-05-11T19:00:00-04:00',
   'startTimezone': 'EDT',
   'streamingURL': 'https://watchufa.tv/videos/montreal-at-toronto-5-11-2024',
   'updateTimestamp': '2024-05-22T23:31:12.385Z',
   'week': 'week-3',
   'location': 'Allan A. Lamport Stadium'},
  {'gameID': '2024-04-27-MIN-PIT',
   'awayTeamID': 'windchill',
   'homeTeamID': 'thunderbirds',
   'awayScore': 18,
   'homeScore': 11,
   'status': 'Final',
   'startTimestamp': '2024-04-27T13:00:00-04:00',
   'startTimezone': 'EDT',
   'streamingURL': 'https://www.watchufa.tv/videos/minnesota-at-pittsburgh-4-27-2024',
   'updateTimestamp': '2024-06-14T02:15:15.104Z',
   'week': 'week-1',
   'location': 'F.N.B. Stadium'},
  {'gameID': '2024-04-27-MTL-NY',
   'awayTeamID': 'royal',
   'homeTeamID': 'empire',
   'awayScore': 16,
   'homeScore': 18

In [10]:
games_json.keys()

dict_keys(['object', 'data'])

In [6]:
games = pd.DataFrame(games_json["data"])
games.head()

,gameID,awayTeamID,homeTeamID,awayScore,homeScore,status,startTimestamp,startTimezone,streamingURL,updateTimestamp,week,location
0,2024-05-11-MTL-TOR,royal,rush,15,12,Final,2024-05-11T19:00:00-04:00,EDT,https://watchufa.tv/videos/montreal-at-toronto...,2024-05-22T23:31:12.385Z,week-3,Allan A. Lamport Stadium
1,2024-04-27-MIN-PIT,windchill,thunderbirds,18,11,Final,2024-04-27T13:00:00-04:00,EDT,https://www.watchufa.tv/videos/minnesota-at-pi...,2024-06-14T02:15:15.104Z,week-1,F.N.B. Stadium
2,2024-04-27-MTL-NY,royal,empire,16,18,Final,2024-04-27T17:00:00-04:00,EDT,https://www.watchufa.tv/videos/montreal-at-new...,2024-05-22T23:31:11.733Z,week-1,Joseph F. Fosina Field
3,2024-04-27-ATL-CAR,hustle,flyers,16,24,Final,2024-04-27T18:00:00-04:00,EDT,https://www.watchufa.tv/videos/atlanta-at-caro...,2024-07-03T20:14:40.764Z,week-1,Durham County Stadium
4,2024-06-08-MAD-DET,radicals,mechanix,19,12,Final,2024-06-08T19:00:00-04:00,EDT,https://watchufa.tv/videos/madison-at-detroit-...,2024-06-16T22:31:01.167Z,week-7,Watervliet High School Stadium


In [7]:
game_id = games.loc[0, "gameID"]

response = requests.get(
    f"{BASE_URL}/gameEvents",
    params={"gameID": game_id}
)

response.raise_for_status()
events_json = response.json()

events_json.keys()

dict_keys(['object', 'data'])

In [12]:
data = events_json["data"]

print(type(data))

if isinstance(data, dict):
    print(data.keys())

    for key, value in data.items():
        print(key, type(value))

<class 'dict'>
dict_keys(['homeEvents', 'awayEvents'])
homeEvents <class 'list'>
awayEvents <class 'list'>


In [13]:
home_events = pd.json_normalize(events_json["data"]["homeEvents"])
away_events = pd.json_normalize(events_json["data"]["awayEvents"])

home_events["team_side"] = "home"
away_events["team_side"] = "away"

events = pd.concat([home_events, away_events], ignore_index=True)

events.head()

,type,line,time,puller,pullX,pullY,pullMs,thrower,throwerX,throwerY,receiver,receiverX,receiverY,defender,turnoverX,turnoverY,team_side
0,1,"[amuraoka, kjay, ktong, lcomire, dmiller2, bkl...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
1,7,NaN,NaN,amuraoka,-9.51,103.52,4843.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
2,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
3,18,NaN,NaN,NaN,NaN,NaN,NaN,lcomire,26.46,72.90,amuraoka,18.49,65.49,NaN,NaN,NaN,home
4,18,NaN,NaN,NaN,NaN,NaN,NaN,amuraoka,18.49,65.49,kjay,4.08,65.69,NaN,NaN,NaN,home


In [14]:
events.shape
events.columns
events.head(20)

,type,line,time,puller,pullX,pullY,pullMs,thrower,throwerX,throwerY,receiver,receiverX,receiverY,defender,turnoverX,turnoverY,team_side
0,1,"[amuraoka, kjay, ktong, lcomire, dmiller2, bkl...",0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
1,7,NaN,NaN,amuraoka,-9.51,103.52,4843.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
2,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,home
3,18,NaN,NaN,NaN,NaN,NaN,NaN,lcomire,26.46,72.90,amuraoka,18.49,65.49,NaN,NaN,NaN,home
4,18,NaN,NaN,NaN,NaN,NaN,NaN,amuraoka,18.49,65.49,kjay,4.08,65.69,NaN,NaN,NaN,home
5,18,NaN,NaN,NaN,NaN,NaN,NaN,kjay,4.08,65.69,dmiller2,-14.04,70.98,NaN,NaN,NaN,home
6,18,NaN,NaN,NaN,NaN,NaN,NaN,dmiller2,-14.04,70.98,amuraoka,1.33,69.95,NaN,NaN,NaN,home
7,18,NaN,NaN,NaN,NaN,NaN,NaN,amuraoka,1.33,69.95,bklar,-12.94,77.84,NaN,NaN,NaN,home
8,18,NaN,NaN,NaN,NaN,NaN,NaN,bklar,-12.94,77.84,dmiller2,-3.81,83.54,NaN,NaN,NaN,home
9,20,NaN,NaN,NaN,NaN,NaN,NaN,dmiller2,-3.81,83.54,lcomire,25.91,93.70,NaN,NaN,NaN,home


In [16]:
events.to_csv(f"../data/raw/game_{game_id}_events.csv", index=False)

In [17]:
for col in events.columns:
    print(col)

type
line
time
puller
pullX
pullY
pullMs
thrower
throwerX
throwerY
receiver
receiverX
receiverY
defender
turnoverX
turnoverY
team_side


In [18]:
events.head().T

,0,1,2,3,4
type,1,7,13,18,18
line,"[amuraoka, kjay, ktong, lcomire, dmiller2, bkl...",NaN,NaN,NaN,NaN
time,0.0,NaN,NaN,NaN,NaN
puller,NaN,amuraoka,NaN,NaN,NaN
pullX,NaN,-9.51,NaN,NaN,NaN
pullY,NaN,103.52,NaN,NaN,NaN
pullMs,NaN,4843.0,NaN,NaN,NaN
thrower,NaN,NaN,NaN,lcomire,amuraoka
throwerX,NaN,NaN,NaN,26.46,18.49
throwerY,NaN,NaN,NaN,72.9,65.49


In [19]:
events["type"].value_counts().sort_index()

type
1      31
2      31
3       6
5       6
7      27
8       3
9       1
11     22
13     32
14      2
15     27
16     11
17     13
18    452
19     27
20      9
22     45
24      3
25      7
28      2
29      2
30      2
31      2
Name: count, dtype: int64

In [20]:
events.head().T

,0,1,2,3,4
type,1,7,13,18,18
line,"[amuraoka, kjay, ktong, lcomire, dmiller2, bkl...",NaN,NaN,NaN,NaN
time,0.0,NaN,NaN,NaN,NaN
puller,NaN,amuraoka,NaN,NaN,NaN
pullX,NaN,-9.51,NaN,NaN,NaN
pullY,NaN,103.52,NaN,NaN,NaN
pullMs,NaN,4843.0,NaN,NaN,NaN
thrower,NaN,NaN,NaN,lcomire,amuraoka
throwerX,NaN,NaN,NaN,26.46,18.49
throwerY,NaN,NaN,NaN,72.9,65.49


In [21]:
throws = events[events["thrower"].notna()].copy()
throws.head().T

,3,4,5,6,7
type,18,18,18,18,18
line,NaN,NaN,NaN,NaN,NaN
time,NaN,NaN,NaN,NaN,NaN
puller,NaN,NaN,NaN,NaN,NaN
pullX,NaN,NaN,NaN,NaN,NaN
pullY,NaN,NaN,NaN,NaN,NaN
pullMs,NaN,NaN,NaN,NaN,NaN
thrower,lcomire,amuraoka,kjay,dmiller2,amuraoka
throwerX,26.46,18.49,4.08,-14.04,1.33
throwerY,72.9,65.49,65.69,70.98,69.95


In [1]:
import sys
sys.path.insert(0, "../src")

from ufa import search_games

games = search_games("2026-06-01", "2026-06-06", team="glory")
games

,gameID,awayTeamID,homeTeamID,awayScore,homeScore,status,startTimestamp,location
2,2026-06-05-NY-BOS,empire,glory,21,17,Final,2026-06-05T19:00:00-04:00,Hormel Stadium
7,2026-06-06-BOS-PHI,glory,phoenix,17,14,Final,2026-06-06T19:00:00-04:00,Neumann University turf field


In [18]:
from ufa import build_game_throws

game_id = "2026-06-05-NY-BOS"
result = build_game_throws(game_id=game_id)

result.thrower_stats

,thrower,attempts,completions,turnovers,huck_attempts,huck_completions,avg_throw_distance,avg_field_y_delta,total_field_y_delta,completion_pct,huck_rate,huck_completion_pct
0,aatkins,38,37,1,6,5,25.114267,12.819737,487.15,0.973684,0.157895,0.833333
1,adavis2,4,4,0,1,1,29.672721,22.365000,89.46,1.000000,0.250000,1.000000
2,ayuan,5,4,1,0,0,16.404370,5.838000,29.19,0.800000,0.000000,NaN
3,beberhard,26,26,0,0,0,16.981244,4.965385,129.10,1.000000,0.000000,NaN
4,bjagt,13,13,0,0,0,15.012259,1.292308,16.80,1.000000,0.000000,NaN
5,bmccann,5,4,1,0,0,15.793784,2.962000,14.81,0.800000,0.000000,NaN
6,bsadok,32,29,3,2,0,19.082971,12.194688,390.23,0.906250,0.062500,0.000000
7,bsimmons,9,9,0,0,0,12.827303,-0.698889,-6.29,1.000000,0.000000,NaN
8,cdavisbra,19,19,0,0,0,12.450006,3.816842,72.52,1.000000,0.000000,NaN
9,cpanarell,5,5,0,0,0,13.739014,-0.694000,-3.47,1.000000,0.000000,NaN
